# YOLO para detectar buracos

Este notebook baixa o dataset do Roboflow, treina um modelo YOLO e gera o arquivo `best.onnx`.

## 1. Instalar bibliotecas

Esta célula instala o que vamos usar no treino, exportação e teste do modelo.

In [ ]:
%pip install -q ultralytics roboflow onnx onnxsim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.0/184.0 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 111.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 142.4 MB/s eta 0:00:00


## 2. Configurar o projeto

Esta célula define sua chave da API, o nome do workspace, o projeto e a versão do dataset.

In [ ]:
from pathlib import Path

ROBOFLOW_API_KEY = "GTkCvnplkh8sKxneIKXS"
WORKSPACE = "abners-lab"
PROJECT = "localizacao-de-buracos-9mb7f-gvqu0"
VERSION = 1

PROJECT_DIR = Path.cwd()
RUNS_DIR = PROJECT_DIR / "runs"
EXPORT_DIR = PROJECT_DIR / "export"
EXPORT_DIR.mkdir(exist_ok=True)

print(f"Pasta do projeto: {PROJECT_DIR}")

Pasta do projeto: /content


In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = "GTkCvnplkh8sKxneIKXS"

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
print(rf.workspace())

loading Roboflow workspace...
{
  "name": "Abners Lab",
  "url": "abners-lab",
  "projects": [
    "abners-lab/lia_project",
    "abners-lab/localizacao-de-buracos-9mb7f-gvqu0",
    "abners-lab/wildfire-smoke-vadpq"
  ]
}


## 3. Baixar o dataset

Esta célula conecta no Roboflow e baixa o dataset no formato YOLO.

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov8")

DATA_YAML = Path(dataset.location) / "data.yaml"
print(f"Dataset salvo em: {dataset.location}")
print(f"Arquivo YAML: {DATA_YAML}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Localização-de-buracos-1 in yolov8:: 100%|██████████| 16633/16633 [00:02<00:00, 5783.03it/s] 


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Dataset salvo em: /content/Localização-de-buracos-1
Arquivo YAML: /content/Localização-de-buracos-1/data.yaml


## 4. Treinar o YOLO

Esta célula usa um modelo pequeno para começar mais rápido e treina com seu dataset.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=16,
    project=str(RUNS_DIR),
    name="buracos"
)

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Localização-de-buracos-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=buracos, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=Tru

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x789166be1130>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

## 5. Localizar o melhor peso

Esta célula pega o arquivo `best.pt`, que é o melhor resultado do treino.

In [ ]:
best_pt = RUNS_DIR / "buracos" / "weights" / "best.pt"
print(f"Melhor peso: {best_pt}")
print(f"Existe? {best_pt.exists()}")

Melhor peso: /content/runs/buracos/weights/best.pt
Existe? True


## 6. Exportar para ONNX

Esta célula converte o melhor modelo para ONNX e copia o arquivo final como `best.onnx`.

In [ ]:
import shutil

best_model = YOLO(str(best_pt))
exported_path = Path(best_model.export(format="onnx", opset=12, simplify=True))

final_onnx = EXPORT_DIR / "best.onnx"
shutil.copy2(exported_path, final_onnx)

print(f"ONNX gerado em: {final_onnx}")

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/runs/buracos/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (6.0 MB)
requirements: Ultralytics requirements ['onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 327ms
Prepared 3 packages in 6.08s
Installed 3 packages in 15ms
 + colorama==0.4.6
 + onnxruntime-gpu==1.25.0
 + onnxslim==0.1.91

requirements: AutoUpdate success ✅ 6.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.21.0 opset 12...
ONNX: slimming with onnxslim 0.1.91...
ONNX: export succe

## 7. Testar em imagem

Esta célula roda o modelo em uma imagem e salva o resultado com as caixas desenhadas.

In [ ]:
IMAGE_PATH = "/content/Captura de tela 2026-04-26 152850.png"

best_model.predict(
    source=IMAGE_PATH,
    conf=0.25,
    save=True,
    project=str(RUNS_DIR),
    name="pred_imagem"
)


image 1/1 /content/Captura de tela 2026-04-26 152850.png: 640x640 3 buracos, 8.9ms
Speed: 4.1ms preprocess, 8.9ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/pred_imagem-4


[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'buraco'}
 obb: None
 orig_img: array([[[177, 176, 176],
         [177, 176, 176],
         [177, 176, 176],
         ...,
         [251, 250, 249],
         [251, 250, 249],
         [251, 250, 249]],
 
        [[251, 250, 249],
         [251, 250, 249],
         [251, 250, 249],
         ...,
         [251, 250, 249],
         [251, 250, 249],
         [251, 250, 249]],
 
        [[251, 250, 249],
         [251, 250, 249],
         [  0,   1,   2],
         ...,
         [132, 133, 131],
         [113, 114, 112],
         [107, 108, 106]],
 
        ...,
 
        [[251, 250, 249],
         [251, 250, 249],
         [ 83,  74,  64],
         ...,
         [177, 168, 155],
         [157, 148, 135],
         [206, 197, 184]],
 
        [[251, 250, 249],
         [251, 250, 249],
         [251, 250, 249],
         ...,
         [251, 250

In [ ]:
from IPython.display import Image, display
from pathlib import Path

saida = RUNS_DIR / "pred_imagem" / Path(IMAGE_PATH).name
print(saida)
display(Image(filename=str(saida)))


/content/runs/pred_imagem/coloque_sua_imagem.jpg


FileNotFoundError: [Errno 2] No such file or directory: '/content/runs/pred_imagem/coloque_sua_imagem.jpg'

## 8. Testar em vídeo

Esta célula roda o modelo em um vídeo e salva um novo vídeo com as detecções.

In [ ]:
VIDEO_PATH = "/content/Buracos_vídeo.mp4"

best_model.predict(
    source=VIDEO_PATH,
    conf=0.25,
    save=True,
    project=str(RUNS_DIR),
    name="pred_video"
)

print("Resultado salvo dentro da pasta runs.")


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/797) /content/Buracos_vídeo.mp4: 640x384 (no detections), 39.2ms
video 1/1 (frame 2/797) /content/Buracos_vídeo.mp4: 640x384 (no detections), 6.6ms
video 1/1 (frame 3/797) /content/Buracos_vídeo.mp4: 640x384 (no detections), 7.2ms
video 1/1 (frame 4/797) /content/Buracos_vídeo.mp4: 640x384 1 buraco, 6.0ms
video 1/1 (frame 5/797) /content/Buracos_vídeo.mp4: 640x384 (no detections), 5.8ms
video 1/1 (frame 6/797) /content/Buracos_vídeo.mp

## 9. Resultado final

Se tudo der certo, o arquivo final ficará em `export/best.onnx`.